In [5]:

import pandas as pd

# 读取原始数据
file_path = r"C:\Users\12635\Documents\NYC_Taxi_Project\data\raw\yellow_tripdata_2026-01.parquet"

df = pd.read_parquet(file_path)

# 创建清洗副本
df_clean = df.copy()

print("数据行数：", df.shape[0])
print("数据列数：", df.shape[1])

df.head()


数据行数： 3724889
数据列数： 20


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [6]:
# 检查数据缺失情况
print("缺失值统计：")
print(df_clean.isnull().sum())

# 检查重复记录
print("\n重复记录数量：")
print(df_clean.duplicated().sum())

# 查看字段类型
print("\n字段类型：")
print(df_clean.dtypes)

缺失值统计：
VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          1088058
trip_distance                  0
RatecodeID               1088058
store_and_fwd_flag       1088058
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     1088058
Airport_fee              1088058
cbd_congestion_fee             0
dtype: int64

重复记录数量：
0

字段类型：
VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID          

In [7]:
# Step 1: 计算每笔行程的持续时间（分钟）

df_clean["trip_duration_minutes"] = (
    df_clean["tpep_dropoff_datetime"]
    - df_clean["tpep_pickup_datetime"]
).dt.total_seconds() / 60

# Step 2: 检查异常时间

negative_duration = (
    df_clean["trip_duration_minutes"] < 0
).sum()

zero_duration = (
    df_clean["trip_duration_minutes"] == 0
).sum()

long_duration = (
    df_clean["trip_duration_minutes"] > 180
).sum()

print("负数行程时长：", negative_duration)
print("零分钟行程：", zero_duration)
print("超过180分钟的行程：", long_duration)

# Step 3: 查看行程时长的统计分布

print("\n行程时长统计：")
print(df_clean["trip_duration_minutes"].describe())

负数行程时长： 1
零分钟行程： 45069
超过180分钟的行程： 1755

行程时长统计：
count    3.724889e+06
mean     1.719316e+01
std      2.517802e+01
min     -1.170000e+01
25%      8.066667e+00
50%      1.336667e+01
75%      2.120000e+01
max      7.507900e+03
Name: trip_duration_minutes, dtype: float64


In [8]:
import numpy as np

# 计算平均速度，避免除以零
df_clean["avg_speed_mph"] = np.where(
    df_clean["trip_duration_minutes"] > 0,
    df_clean["trip_distance"] /
    (df_clean["trip_duration_minutes"] / 60),
    np.nan
)

# 检查距离异常
print("零距离行程：", (df_clean["trip_distance"] == 0).sum())
print("负距离行程：", (df_clean["trip_distance"] < 0).sum())
print("超过100英里的行程：", (df_clean["trip_distance"] > 100).sum())

# 检查速度异常
print("超过100 mph的行程：", (df_clean["avg_speed_mph"] > 100).sum())

# 交叉检查：零分钟但距离大于0
print(
    "零分钟但距离大于0：",
    (
        (df_clean["trip_duration_minutes"] == 0) &
        (df_clean["trip_distance"] > 0)
    ).sum()
)

# 查看分布
print("\n距离统计：")
print(df_clean["trip_distance"].describe())

print("\n速度统计：")
print(df_clean["avg_speed_mph"].describe())

零距离行程： 125738
负距离行程： 0
超过100英里的行程： 162
超过100 mph的行程： 732
零分钟但距离大于0： 43905

距离统计：
count    3.724889e+06
mean     6.455647e+00
std      6.488855e+02
min      0.000000e+00
25%      1.000000e+00
50%      1.810000e+00
75%      3.730000e+00
max      2.690975e+05
Name: trip_distance, dtype: float64

速度统计：
count    3.679819e+06
mean     2.426281e+01
std      3.357625e+03
min      0.000000e+00
25%      6.737615e+00
50%      9.253099e+00
75%      1.294274e+01
max      2.218900e+06
Name: avg_speed_mph, dtype: float64


In [9]:
# Step 1: 为每条记录添加数据质量标记

df_clean["is_valid_trip"] = (
    (df_clean["trip_duration_minutes"] > 0) &
    (df_clean["trip_duration_minutes"] <= 180) &
    (df_clean["trip_distance"] > 0) &
    (df_clean["trip_distance"] <= 100) &
    (df_clean["avg_speed_mph"] > 0) &
    (df_clean["avg_speed_mph"] <= 100)
)

# Step 2: 统计有效和异常记录
print("原始记录数：", len(df_clean))
print("有效记录数：", df_clean["is_valid_trip"].sum())
print("异常记录数：", (~df_clean["is_valid_trip"]).sum())

print("\n有效记录比例：")
print(df_clean["is_valid_trip"].value_counts(normalize=True) * 100)

# Step 3: 创建用于分析的 DWD 数据集
df_dwd = df_clean.loc[df_clean["is_valid_trip"]].copy()

print("\nDWD 数据规模：", df_dwd.shape)

原始记录数： 3724889
有效记录数： 3552907
异常记录数： 171982

有效记录比例：
is_valid_trip
True     95.382896
False     4.617104
Name: proportion, dtype: float64

DWD 数据规模： (3552907, 23)


In [10]:
# Step 4: 金额字段质量检查

amount_columns = [
    "fare_amount",
    "total_amount",
    "tip_amount",
    "tolls_amount"
]

for col in amount_columns:
    print(f"\n===== {col} =====")
    print("负金额记录：", (df_clean[col] < 0).sum())
    print("零金额记录：", (df_clean[col] == 0).sum())
    print("超过500美元：", (df_clean[col] > 500).sum())

    print(df_clean[col].describe(
        percentiles=[0.5, 0.95, 0.99, 0.999]
    ))


===== fare_amount =====
负金额记录： 39463
零金额记录： 2082
超过500美元： 59
count    3.724889e+06
mean     2.080425e+01
std      1.892701e+01
min     -2.555200e+03
50%      1.560000e+01
95%      5.850000e+01
99%      8.140000e+01
99.9%    1.458000e+02
max      2.555200e+03
Name: fare_amount, dtype: float64

===== total_amount =====
负金额记录： 39984
零金额记录： 433
超过500美元： 84
count    3.724889e+06
mean     2.917853e+01
std      2.258553e+01
min     -2.560200e+03
50%      2.305000e+01
95%      7.525000e+01
99%      1.047500e+02
99.9%    1.755145e+02
max      2.560200e+03
Name: total_amount, dtype: float64

===== tip_amount =====
负金额记录： 67
零金额记录： 1597981
超过500美元： 3
count    3.724889e+06
mean     2.608142e+00
std      3.917405e+00
min     -8.888000e+01
50%      2.000000e+00
95%      1.000000e+01
99%      1.729000e+01
99.9%    2.832112e+01
max      7.660000e+02
Name: tip_amount, dtype: float64

===== tolls_amount =====
负金额记录： 3391
零金额记录： 3484180
超过500美元： 0
count    3.724889e+06
mean     4.983751e-01
std      2.1

In [11]:
# Step 5: 金额数据质量标记

df_clean["is_negative_amount"] = (
    df_clean["total_amount"] < 0
)

df_clean["is_zero_amount"] = (
    df_clean["total_amount"] == 0
)

df_clean["is_high_amount"] = (
    df_clean["total_amount"] > 500
)

# 收入分析专用标记
df_clean["is_valid_revenue"] = (
    df_clean["is_valid_trip"] &
    (df_clean["total_amount"] > 0) &
    (df_clean["total_amount"] <= 500)
)

# 查看统计结果
print("负金额交易：", df_clean["is_negative_amount"].sum())
print("零金额交易：", df_clean["is_zero_amount"].sum())
print("高金额交易：", df_clean["is_high_amount"].sum())

print("\n可用于常规收入分析的记录：")
print(df_clean["is_valid_revenue"].sum())

print("\n收入分析有效比例：")
print(
    round(df_clean["is_valid_revenue"].mean() * 100, 2),
    "%"
)

负金额交易： 39984
零金额交易： 433
高金额交易： 84

可用于常规收入分析的记录：
3515574

收入分析有效比例：
94.38 %


In [12]:
from pathlib import Path
import pandas as pd

# 项目根目录
project_dir = Path(
    r"C:\Users\12635\Documents\NYC_Taxi_Project"
)

# 创建 DWD 输出目录
dwd_dir = project_dir / "data" / "processed" / "dwd"
dwd_dir.mkdir(parents=True, exist_ok=True)

# 保留全部记录及质量标记
df_dwd_all = df_clean.copy()

# 生成 DWD 文件
output_file = dwd_dir / "yellow_taxi_dwd_2026-01.parquet"

df_dwd_all.to_parquet(
    output_file,
    engine="pyarrow",
    index=False
)

print("DWD 文件保存成功！")
print("文件位置：", output_file)
print("记录数：", len(df_dwd_all))
print("字段数：", len(df_dwd_all.columns))

DWD 文件保存成功！
文件位置： C:\Users\12635\Documents\NYC_Taxi_Project\data\processed\dwd\yellow_taxi_dwd_2026-01.parquet
记录数： 3724889
字段数： 27


In [13]:
# 创建质量报告
quality_metrics = {
    "total_records": len(df_clean),
    "valid_trip_records": int(df_clean["is_valid_trip"].sum()),
    "invalid_trip_records": int((~df_clean["is_valid_trip"]).sum()),
    "valid_revenue_records": int(df_clean["is_valid_revenue"].sum()),
    "negative_duration": int(
        (df_clean["trip_duration_minutes"] < 0).sum()
    ),
    "zero_duration": int(
        (df_clean["trip_duration_minutes"] == 0).sum()
    ),
    "zero_distance": int(
        (df_clean["trip_distance"] == 0).sum()
    ),
    "high_speed": int(
        (df_clean["avg_speed_mph"] > 100).sum()
    ),
    "negative_amount": int(
        df_clean["is_negative_amount"].sum()
    ),
    "zero_amount": int(
        df_clean["is_zero_amount"].sum()
    ),
    "high_amount": int(
        df_clean["is_high_amount"].sum()
    )
}

quality_report = pd.DataFrame(
    [
        {
            "metric": key,
            "value": value
        }
        for key, value in quality_metrics.items()
    ]
)

report_file = dwd_dir / "taxi_data_quality_report_2026-01.csv"

quality_report.to_csv(
    report_file,
    index=False,
    encoding="utf-8-sig"
)

print("数据质量报告保存成功！")
print("报告位置：", report_file)

display(quality_report)

数据质量报告保存成功！
报告位置： C:\Users\12635\Documents\NYC_Taxi_Project\data\processed\dwd\taxi_data_quality_report_2026-01.csv


,metric,value
0,total_records,3724889
1,valid_trip_records,3552907
2,invalid_trip_records,171982
3,valid_revenue_records,3515574
4,negative_duration,1
5,zero_duration,45069
6,zero_distance,125738
7,high_speed,732
8,negative_amount,39984
9,zero_amount,433


In [14]:
import pandas as pd

df_check = pd.read_parquet(output_file)

print("DWD 记录数：", len(df_check))
print("DWD 字段数：", len(df_check.columns))
print("有效行程：", df_check["is_valid_trip"].sum())
print("有效收入记录：", df_check["is_valid_revenue"].sum())

assert len(df_check) == 3724889
assert df_check["is_valid_trip"].sum() == 3552907
assert df_check["is_valid_revenue"].sum() == 3515574

print("DWD 数据验证通过！")

DWD 记录数： 3724889
DWD 字段数： 27
有效行程： 3552907
有效收入记录： 3515574
DWD 数据验证通过！
